In [1]:
import pandas as pd
import os
from tqdm import tqdm
from openai import OpenAI

ITEM_META_PATH = "./movie_book_cdsr_processed/item_meta.csv"
SAVE_PATH = "./movie_book_cdsr_processed/item_meta_llm_enhanced.csv"
BATCH_SIZE = 500
TEMPERATURE = 0.1
MAX_TOKENS = 128

In [4]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="local_gpu"
)

def build_prompt(domain, title):
    domain_name = "Movie" if domain == 0 else "Book"
    prompt = f"""Generate additional contextual information for an item of type {domain_name} with title "{title}". 
Provide detailed insights and keywords about the item’s features. 
The output should include specific attributes and potential users that enhance cross-domain recommendation.
Output only structured content, no extra words.
"""
    return prompt

if __name__ == "__main__":
    # 读取数据 + 断点续跑
    item_meta = pd.read_csv(ITEM_META_PATH)
    print(f"总商品数：{len(item_meta)}")

    if os.path.exists(SAVE_PATH):
        print("找到历史结果，断点续跑")
        saved_df = pd.read_csv(SAVE_PATH)
        item_meta = pd.merge(
            item_meta,
            saved_df[["item_id", "llm_enhanced_text"]],
            on="item_id",
            how="left"
        )
    else:
        item_meta["llm_enhanced_text"] = ""

    # 筛选未处理的数据
    to_process = item_meta[
        item_meta["llm_enhanced_text"].isna() | 
        (item_meta["llm_enhanced_text"] == "")
    ]
    print(f"待处理商品数：{len(to_process)}")

    if len(to_process) == 0:
        print("全部生成完成！")
        exit()

    print("显卡版 Qwen-0.5B 开始生成...")
    for idx, row in tqdm(enumerate(to_process.itertuples(index=False)), total=len(to_process)):
        item_id = row.item_id
        prompt = build_prompt(row.domain, row.title)
        enhanced_text = ""

        try:
            # 显卡极速调用
            response = client.chat.completions.create(
                model="qwen:0.5b",
                messages=[
                    {"role": "system", "content": "You are a professional cross-domain item analyst."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE
            )
            enhanced_text = response.choices[0].message.content.strip()
        except Exception as e:
            print(f"失败 item_id:{item_id}")

        # 写入结果
        item_meta.loc[item_meta["item_id"] == item_id, "llm_enhanced_text"] = enhanced_text

        # 批量保存
        if idx % BATCH_SIZE == 0 and idx > 0:
            item_meta.to_csv(SAVE_PATH, index=False, encoding="utf-8")

    # 最终保存
    item_meta.to_csv(SAVE_PATH, index=False, encoding="utf-8")
    print("全部完成！文件已保存")

总商品数：95661
找到历史结果，断点续跑
待处理商品数：13151
显卡版 Qwen-0.5B 开始生成...


100%|██████████| 13151/13151 [1:36:59<00:00,  2.26it/s] 


全部完成！文件已保存
